# Lezione 2 — Duplicati, tipi errati e outlier

**Come si usa questo notebook.** Leggi ogni cella, prova l'esercizio prima
di aprire la soluzione, e alla fine fai crescere il progetto. Prerequisito:
aver eseguito la Lezione 1, che copriva i valori mancanti (campi critici da
scartare, misure recuperabili da imputare con un flag) e ha salvato
`datasets/processed/memory_events_clean.csv` — questo notebook riparte da
quel file.

**Cosa saprai fare alla fine:** riconoscere e gestire i tre difetti più
frequenti di qualunque tabella reale — righe che contano due volte, numeri
letti come testo, valori impossibili — senza nascondere le correzioni.

## Parte 1 — Teoria: l'identità di un record (duplicati)

**Perché è un problema generale.** Una riga contata due volte pesa doppio in
ogni statistica, in ogni training set, in ogni fattura. Succede ovunque: un
retry di rete reinvia lo stesso dato, un utente clicca due volte "invia",
due sistemi esportano lo stesso evento.

Il concetto teorico centrale: **"duplicato" non è una proprietà della riga,
è una decisione tua su cosa significa identità**. Devi dichiarare una
*chiave*: quali colonne, insieme, identificano un'entità del mondo reale?

- **Duplicato esatto**: stessa chiave, valori identici. Facile.
- **Near-duplicate**: stessa entità scritta in modo diverso — `"north"` e
  `"  NORTH  "`, "Via Roma 1" e "via roma 1". È il caso realistico, e il
  campo di studio si chiama *record linkage*: ogni confronto approssimato
  bilancia **falsi match** (unisco cose diverse) e **match mancati**
  (non unisco la stessa cosa). Chaudhuri et al. (2003) formalizzano questo
  trade-off: la somiglianza **propone candidati**, non dimostra identità.

Il trade-off si vede bene con un caso limite. Confronta due normalizzazioni
diverse per lo stesso paio di stazioni:

- normalizzazione **debole** (solo minuscole): `"north"` e `"North Station"`
  restano diverse — un vero duplicato *sfugge* (match mancato: costo basso,
  ma qualche riga doppia resta nascosta nei dati);
- normalizzazione **aggressiva** (rimuovi anche gli spazi e le parole
  generiche come "station"): ora `"north"` e `"north station"` combaciano —
  ma con questa stessa regola anche `"north gate"` e `"north side"`
  potrebbero collassare insieme, se sono due stazioni realmente diverse che
  condividono solo la parola "north" (falso match: unisci per errore due
  entità distinte, perdendo dati veri).

Non esiste una normalizzazione "giusta" in assoluto: più è aggressiva, meno
duplicati veri sfuggono, ma più rischi di fondere entità diverse. La scelta
dipende da quanto costa, nel tuo dominio, ciascuno dei due errori — ed è
per questo che la normalizzazione va sempre **dichiarata esplicitamente**
nel codice, mai lasciata implicita.

Regola pratica trasferibile: normalizza in modo dichiarato (spazi,
maiuscole), confronta sulla chiave scelta, conserva la prima occorrenza, e
**conta cosa rimuovi** — il conteggio è anche il modo in cui, rivedendo il
report, puoi accorgerti se la normalizzazione scelta è stata troppo debole
o troppo aggressiva.

## Parte 2 — Teoria: tipi errati e outlier

**Tipi.** Un CSV è testo: `18.4` e `"18.4"` e `"sensor_error"` viaggiano
uguali. Se una sola cella non è numerica, l'intera colonna diventa testo e
ogni calcolo tace o sbaglia. La conversione deve essere **esplicita**
(`pd.to_numeric(..., errors='coerce')`): i parse falliti diventano `NaN`
e tu li **flagghi** — non li lasci sparire in silenzio.

**Outlier.** Due concetti diversi che non vanno confusi:

1. **Outlier statistico** — insolito rispetto alla distribuzione, secondo
   una regola dichiarata (la classica: fuori da
   [Q1 − 1.5·IQR, Q3 + 1.5·IQR]). È un *candidato da investigare*, non un
   errore: un valore raro ma legittimo non si "corregge".
2. **Outlier di dominio** — viola un **contratto esterno**: il sensore è
   certificato da −50 a 60 °C, quindi 79 °C è fisicamente invalido anche se
   comparisse spesso.

Per gli outlier di dominio una correzione comune è il **clipping** (riporto
al confine). Attenzione, vale ovunque: il clipping accumula massa sui bordi
e cambia forma e varianza della distribuzione — si usa solo con un
contratto motivato e un flag di audit. L'alternativa (scartare la riga) si
valuta caso per caso.

Vediamo tutto su un esempio minimo:

In [ ]:
import pandas as pd

demo = pd.DataFrame({
    'station_id':    ['north', 'north', '  NORTH  ', 'south'],
    'recorded_at':   ['10:00', '10:00', '10:00',     '10:00'],
    'temperature_c': [18.4,    18.4,    '18.4',      'guasto'],
})

# 1) duplicati esatti sulla chiave dichiarata (stazione + istante)
print('Duplicati esatti :', demo.duplicated(['station_id', 'recorded_at'], keep='first').tolist())

# 2) near-duplicates: dopo la normalizzazione la riga 2 diventa candidata
demo['station_norm'] = demo['station_id'].str.strip().str.casefold()
print('Dopo normalizz.  :', demo.duplicated(['station_norm', 'recorded_at'], keep='first').tolist())

# 3) tipi: la conversione esplicita rende visibili i parse falliti
numeri = pd.to_numeric(demo['temperature_c'], errors='coerce')
print('Parse falliti    :', (demo['temperature_c'].notna() & numeri.isna()).tolist())

Nota il passaggio chiave: la riga con `'  NORTH  '` era invisibile al
controllo esatto e diventa candidata dopo la normalizzazione. E `'guasto'`
non ha rotto niente: è diventato un `NaN` flaggabile.

E la regola IQR per gli outlier statistici, su numeri veri:

In [ ]:
serie = pd.Series([17.8, 18.1, 18.4, 18.9, 19.2, 19.5, 34.0])
q1, q3 = serie.quantile(0.25), serie.quantile(0.75)
iqr = q3 - q1
basso, alto = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f'Intervallo IQR: [{basso:.1f}, {alto:.1f}]')
print('Candidati outlier statistici:', serie[(serie < basso) | (serie > alto)].tolist())

34.0 è un *candidato*: la regola lo segnala, ma se il sensore è certificato
fino a 60 °C non hai il diritto di "correggerlo" — solo di investigarlo.
La correzione automatica è legittima solo quando c'è un **contratto di
dominio** violato. Tieni questa distinzione: ti servirà identica con le
metriche dei modelli.

## Parte 3 — Esercizio guidato

Stesso dataset di sensori del corso, versione con problemi di qualità:
120 letture, e **non ti dico quanti problemi ci sono né dove**. Contratto
di dominio dichiarato: temperatura valida tra **−50 e 60 °C**.

Compiti, nei termini della teoria:

1. trova i **candidati duplicati** con tre controlli: stesso `reading_id`;
   stessa coppia (`station_id`, `recorded_at`); stessa coppia dopo la
   **normalizzazione** delle etichette;
2. rimuovi i candidati (conserva la prima occorrenza) e conta;
3. converti `temperature_c` esplicitamente e **flagga** i parse falliti;
4. flagga i valori fuori contratto e applica il **clipping** a [−50, 60];
5. produci il report delle correzioni.

Partenza:

In [ ]:
from pathlib import Path

raw = pd.read_csv(Path('..') / 'datasets' / 'synthetic' / 'environmental_sensor_quality_challenge.csv')
print(raw.shape)
raw.head()

**Prova tu nella cella sotto.** Schema: normalizza → tre maschere
`duplicated` combinate con `|` → rimuovi → `to_numeric` + flag → mediana dei
valori validi nel dominio per i parse falliti → flag outlier → `clip`.

In [ ]:
audit = raw.copy()

# Scrivi qui il tuo tentativo:
# 1) normalizza station_id (strip + casefold) in una colonna d'appoggio
# 2) tre maschere duplicated (reading_id; station+istante; normalizzata+istante)
# 3) rimuovi i candidati, conserva la prima occorrenza
# 4) to_numeric con errors='coerce' + flag temperature_was_invalid_type
# 5) flag temperature_was_domain_outlier e clip a [-50, 60]

audit.head()

### Soluzione spiegata

- le tre maschere si combinano con `|`: una riga è candidata se **almeno un**
  controllo la segnala;
- la mediana di riempimento si calcola **solo sui valori validi e nel
  dominio**: usare anche i fuori-range inquinerebbe la statistica;
- il flag outlier va creato **prima** del `clip`, o non saprai più quali
  valori hai toccato — stesso principio del flag di imputazione della
  Lezione 1: se registri "questo valore è stato modificato" solo *dopo*
  aver già modificato il valore, il flag risulterebbe sempre falso, perché
  a quel punto il valore originale non esiste già più;
- il report chiude il cerchio: ogni correzione è contata.

In [ ]:
soluzione = raw.copy()
righe_prima = len(soluzione)

norm = soluzione['station_id'].astype('string').str.strip().str.casefold()
candidati = (
    soluzione.duplicated('reading_id', keep='first')
    | soluzione.duplicated(['station_id', 'recorded_at'], keep='first')
    | soluzione.assign(_n=norm).duplicated(['_n', 'recorded_at'], keep='first')
)
soluzione = soluzione.loc[~candidati].reset_index(drop=True)

numeri = pd.to_numeric(soluzione['temperature_c'], errors='coerce')
soluzione['temperature_was_invalid_type'] = soluzione['temperature_c'].notna() & numeri.isna()
mediana_valida = numeri[numeri.between(-50, 60)].median()
soluzione['temperature_c'] = numeri.fillna(mediana_valida)

soluzione['temperature_was_domain_outlier'] = ~soluzione['temperature_c'].between(-50, 60)
soluzione['temperature_c'] = soluzione['temperature_c'].clip(-50, 60)

report = {
    'righe_prima': righe_prima,
    'candidati_duplicati_rimossi': int(candidati.sum()),
    'parse_falliti': int(soluzione['temperature_was_invalid_type'].sum()),
    'outlier_di_dominio': int(soluzione['temperature_was_domain_outlier'].sum()),
    'righe_finali': len(soluzione),
}
assert soluzione['reading_id'].is_unique
assert soluzione['temperature_c'].between(-50, 60).all()
report

## Parte 4 — Il progetto: Memory AI Lab, passo 2

È arrivato un **nuovo batch di memorie** da integrare nell'archivio che hai
creato nella Lezione 1 (il file `memory_events_clean.csv`: memorie senza
buchi nei campi critici, con `type` e `importance` già imputati dove
mancavano) — e porta con sé esattamente i difetti di oggi: retry
di ingestion che duplicano record, un parser che ha scritto `high` al posto
di un numero, punteggi fuori dal contratto (per `importance` il contratto è
**0–1**).

Il passo di oggi: **unire il nuovo batch all'archivio, applicando le tre
difese**.

In [ ]:
archivio = pd.read_csv(Path('..') / 'datasets' / 'processed' / 'memory_events_clean.csv')
nuovo_batch = pd.read_csv(Path('..') / 'datasets' / 'synthetic' / 'memory_events_quality_issues.csv')

print(f'Archivio esistente: {len(archivio)} memorie   Nuovo batch: {len(nuovo_batch)} memorie')
nuovo_batch

In [ ]:
memorie = pd.concat([archivio, nuovo_batch], ignore_index=True)
righe_prima = len(memorie)

# Difesa 1: candidati duplicati (id; testo+istante; testo normalizzato+istante)
testo_norm = memorie['text'].astype('string').str.strip().str.casefold()
candidati = (
    memorie.duplicated('memory_id', keep='first')
    | memorie.duplicated(['text', 'timestamp'], keep='first')
    | memorie.assign(_n=testo_norm).duplicated(['_n', 'timestamp'], keep='first')
)
memorie = memorie.loc[~candidati].reset_index(drop=True)

# Difesa 2: importance deve essere numerica; i parse falliti restano visibili
numeri = pd.to_numeric(memorie['importance'], errors='coerce')
memorie['importance_was_invalid_type'] = memorie['importance'].notna() & numeri.isna()
mediana_valida = numeri[numeri.between(0, 1)].median()
memorie['importance'] = numeri.fillna(mediana_valida)

# Difesa 3: contratto di dominio 0-1, flag prima del clip
memorie['importance_was_outlier'] = ~memorie['importance'].between(0, 1)
memorie['importance'] = memorie['importance'].clip(0, 1)

# i flag della lezione 1 restano; per il nuovo batch valgono False
for flag in ['type_was_missing', 'importance_was_missing']:
    memorie[flag] = memorie[flag].fillna(False)
memorie['type'] = memorie['type'].fillna('unknown')

destinazione = Path('..') / 'datasets' / 'processed' / 'memory_events_clean.csv'
memorie.to_csv(destinazione, index=False)

print(f'Memorie totali ricevute : {righe_prima}')
print(f'Duplicati rimossi       : {int(candidati.sum())}')
print(f"Parse falliti (flag)    : {int(memorie['importance_was_invalid_type'].sum())}")
print(f"Fuori contratto (flag)  : {int(memorie['importance_was_outlier'].sum())}")
print(f'Archivio aggiornato     : {len(memorie)} memorie')
memorie

L'archivio del progetto ora ha ingestion (Lezione 1) **e** controllo qualità
(Lezione 2): ogni memoria è unica, numerica dove deve esserlo, dentro i
contratti, e ogni correzione è flaggata. Su questa base, nelle prossime
lezioni, costruiremo split corretti e poi le prime feature per i modelli.

## Cosa hai imparato

- L'**identità** di un record è una decisione (la chiave), non un fatto: i
  near-duplicates si trovano normalizzando, e la somiglianza propone
  candidati, non certezze.
- La conversione dei tipi va fatta in modo **esplicito**: i parse falliti
  diventano flag, non sparizioni silenziose.
- **Outlier statistico** (candidato da investigare) ≠ **outlier di dominio**
  (violazione di contratto): solo il secondo si corregge automaticamente.
- Il **clipping** altera la distribuzione: contratto motivato + flag,
  sempre.
- Il progetto cresce per **strati tracciabili**: ogni lezione una difesa.

## Quiz

1. Una temperatura di 51 °C compare una sola volta in tutto il dataset. La
   regola IQR la segnala. Va corretta?
2. Perché la mediana di riempimento per i parse falliti si calcola solo sui
   valori dentro il contratto di dominio?
3. Nel progetto, perché il flag `importance_was_outlier` va creato prima
   del `clip`?

<details>
<summary><b>Apri le risposte</b></summary>

1. No. È un outlier statistico (raro) ma dentro il contratto di dominio
   (−50..60): è un candidato da investigare, non un errore da correggere.
   Correggerlo cancellerebbe forse l'evento più interessante del dataset.
2. Perché i valori fuori contratto sono essi stessi sospetti: se 79 °C
   entrasse nella mediana, il valore di riempimento sarebbe inquinato da un
   dato invalido. La statistica di recupero si calcola solo su dati fidati.
3. Dopo il `clip` tutti i valori sono dentro 0–1 per costruzione: non
   esisterebbe più alcun modo di sapere quali erano fuori. Il flag creato
   prima conserva l'informazione per audit e analisi future.
</details>

## Prossima lezione

L'archivio è pulito: ora va **diviso** senza barare. Train, validation e
test — e il primo incontro con l'errore più subdolo del machine learning:
il data leakage.